In [19]:
import json
import os
import subprocess
from pathlib import Path
from typing import Any

from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool
from langchain_ollama import ChatOllama



In [20]:
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://10.42.0.247:11434/",
)

MODEL_NAME = os.getenv(
    "OLLAMA_MODEL",
    "gpt-oss:20b",
)

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.1,
    num_ctx=8192,
)

print(f"Using {MODEL_NAME} at {OLLAMA_BASE_URL}")

Using gpt-oss:20b at http://10.42.0.247:11434/


In [21]:
response = llm.invoke("Reply with exactly: Ollama connection works")
print(response.content)


Ollama connection works


In [22]:
WORKSPACE = Path("./agent_loop_output/agent_project1")
WORKSPACE.mkdir(parents=True, exist_ok=True)

WORKSPACE = WORKSPACE.resolve()
print(WORKSPACE)

/localjup/notebooks/agent_loop_output/agent_project1


In [23]:
def safe_path(relative_path: str) -> Path:
    """
    Resolve a user-provided path inside WORKSPACE.
    Prevents paths such as ../../etc/passwd.
    """
    path = (WORKSPACE / relative_path).resolve()

    if path != WORKSPACE and WORKSPACE not in path.parents:
        raise ValueError(f"Path escapes workspace: {relative_path}")

    return path

In [24]:
@tool
def list_files() -> str:
    """List files and directories in the current project workspace."""
    entries = []

    for path in sorted(WORKSPACE.rglob("*")):
        relative = path.relative_to(WORKSPACE)
        if ".git" in relative.parts or "__pycache__" in relative.parts:
            continue

        suffix = "/" if path.is_dir() else ""
        entries.append(f"{relative}{suffix}")

    return "\n".join(entries) if entries else "(workspace is empty)"


In [25]:
@tool
def read_file(path: str) -> str:
    """Read a UTF-8 text file from the project workspace."""
    file_path = safe_path(path)

    if not file_path.exists():
        return f"File does not exist: {path}"

    if not file_path.is_file():
        return f"Not a file: {path}"

    content = file_path.read_text(encoding="utf-8")

    # Avoid flooding the model context with very large files.
    max_chars = 30_000
    if len(content) > max_chars:
        content = content[:max_chars] + "\n...[truncated]"

    return content


In [26]:
from importlib.metadata import version

for package in [
    "langchain",
    "langchain-core",
    "langchain-ollama",
]:
    print(package, version(package))

langchain 1.3.17
langchain-core 1.6.0
langchain-ollama 1.1.0


In [27]:
@tool
def write_file(path: str, content: str) -> str:
    """Create a new UTF-8 text file inside the project workspace."""
    file_path = safe_path(path)

    if file_path.exists():
        return (
            f"File already exists: {path}. "
            "Use read_file followed by edit_file to modify it."
        )

    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding="utf-8")

    return (
        f"Created {len(content)} characters in "
        f"{file_path.relative_to(WORKSPACE)}"
    )


In [28]:
@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """
    Edit a UTF-8 text file by replacing one exact occurrence of old_text
    with new_text.

    The file must already exist. The replacement is intentionally limited
    to one occurrence so the agent cannot accidentally modify multiple
    unrelated sections.
    """
    file_path = safe_path(path)

    if not file_path.exists():
        return f"File does not exist: {path}"

    if not file_path.is_file():
        return f"Not a file: {path}"

    content = file_path.read_text(encoding="utf-8")

    occurrences = content.count(old_text)

    if occurrences == 0:
        return (
            f"Could not edit {path}: old_text was not found. "
            "Read the file again and use an exact text match."
        )

    if occurrences > 1:
        return (
            f"Could not edit {path}: old_text occurs {occurrences} times. "
            "Provide a larger, more specific old_text block."
        )

    updated_content = content.replace(old_text, new_text, 1)
    file_path.write_text(updated_content, encoding="utf-8")

    return (
        f"Edited {file_path.relative_to(WORKSPACE)}: "
        f"replaced {len(old_text)} characters with {len(new_text)} characters."
    )


In [29]:
@tool
def run_command(command: str) -> str:
    """
    Run a non-interactive shell command inside the project workspace.
    Use this for formatting, tests, compilation, and inspection.
    """
    blocked_fragments = [
        "rm -rf",
        "shutdown",
        "reboot",
        "mkfs",
        "dd if=",
        ":(){",
        "curl | sh",
        "wget | sh",
    ]

    normalized = command.lower().replace(" ", "")
    for fragment in blocked_fragments:
        if fragment.replace(" ", "") in normalized:
            return f"Blocked potentially destructive command: {command}"

    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=WORKSPACE,
            capture_output=True,
            text=True,
            timeout=60,
            env={
                **os.environ,
                "PYTHONUNBUFFERED": "1",
            },
        )

        output = (
            f"exit_code: {result.returncode}\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )

        if len(output) > 20_000:
            output = output[:20_000] + "\n...[output truncated]"

        return output

    except subprocess.TimeoutExpired:
        return "Command timed out after 60 seconds."
    except Exception as exc:
        return f"Command failed to run: {type(exc).__name__}: {exc}"


In [33]:
TOOLS = [
    list_files,
    read_file,
    write_file,
    edit_file,
    run_command,
]


TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}

for item in TOOLS:
    print(item.name)


list_files
read_file
write_file
edit_file
run_command


In [34]:
llm_with_tools = llm.bind_tools(TOOLS)


In [35]:
response = llm_with_tools.invoke(
    "Use the list_files tool and report the files in the workspace."
)

print("content:", response.content)
print("tool calls:", response.tool_calls)

content: 
tool calls: [{'name': 'list_files', 'args': {}, 'id': '1b85b76b-dc45-4210-9431-8f086d4bff0f', 'type': 'tool_call'}]


In [50]:
SYSTEM_PROMPT = """
You are an expert web‑developer and AI assistant.  
Your task is to produce complete, production‑ready code for a very lightweight WebSocket‑based proximity chat system that can be embedded in a Three.js / WebXR VR world.  
The system must run on PHP 7.3+ (using Ratchet) and vanilla JavaScript.  
The code should be self‑contained, with clear file names and minimal external dependencies.  

Rules:
- Inspect existing files before modifying them.
- Use write_file to create new files.
- Use edit_file for targeted modifications to existing files.
- Use read_file to inspect relevant files before editing them.
- When using edit_file, provide an exact old_text match and a precise new_text replacement.
- If an edit_file operation fails because old_text was not found or is ambiguous, read the file again before retrying.
- Use run_command for tests, formatters, linters, compilers, and basic inspection.
- Do not claim that code works unless you actually run an appropriate check.
- Keep generated code focused and maintainable.
- Ask for clarification only when the requirement is genuinely ambiguous.
- Do not delete or overwrite unrelated files.
- All paths must be relative to the project workspace.
- When providing any text, double‑escape backslashes: `\\\\` instead of `\\`.  
"""

In [51]:
print(SYSTEM_PROMPT)


You are an expert web‑developer and AI assistant.  
Your task is to produce complete, production‑ready code for a very lightweight WebSocket‑based proximity chat system that can be embedded in a Three.js / WebXR VR world.  
The system must run on PHP 7.3+ (using Ratchet) and vanilla JavaScript.  
The code should be self‑contained, with clear file names and minimal external dependencies.  

Rules:
- Inspect existing files before modifying them.
- Use write_file to create new files.
- Use edit_file for targeted modifications to existing files.
- Use read_file to inspect relevant files before editing them.
- When using edit_file, provide an exact old_text match and a precise new_text replacement.
- If an edit_file operation fails because old_text was not found or is ambiguous, read the file again before retrying.
- Use run_command for tests, formatters, linters, compilers, and basic inspection.
- Do not claim that code works unless you actually run an appropriate check.
- Keep generated 

In [52]:
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

def run_agent(
    user_request: str,
    max_iterations: int = 50,
    verbose: bool = True,
) -> str:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_request),
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- iteration {iteration + 1} ---")

        response = llm_with_tools.invoke(messages)
        messages.append(response)

        tool_calls = response.tool_calls or []

        if verbose:
            if response.content:
                print("Assistant:", response.content)
            print("Tool calls:", tool_calls)

        # The model is finished when it returns no tool calls.
        if not tool_calls:
            return response.content or "(no final response)"

        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("args", {})
            tool_call_id = tool_call["id"]

            selected_tool = TOOLS_BY_NAME.get(tool_name)

            if selected_tool is None:
                tool_result = f"Unknown tool: {tool_name}"
            else:
                try:
                    tool_result = selected_tool.invoke(tool_args)
                except Exception as exc:
                    tool_result = (
                        f"Tool error: {type(exc).__name__}: {exc}"
                    )

            if verbose:
                print(f"Executing: {tool_name}({tool_args})")
                print(str(tool_result)[:2_000])

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call_id,
                )
            )

    return (
        f"Agent stopped after {max_iterations} iterations. "
        "The workspace may contain partial results."
    )


In [53]:
request = """
 ### Context
 - The server should accept WebSocket connections from clients and keep all state only in memory (no disk or DB writes).  
 - Each client first sends a JSON control message `{ "type": "register", "id": "<unique‑user‑hash>" }`.  
 - Clients then periodically (≈100 ms) send `{ "type": "position", "pos": { "x":…, "y":…, "z":… } }`.  
 - Clients record audio using MediaRecorder (`audio/webm; codecs=opus`) with a 50 ms chunk size (≈64 kbps).  
 - The server receives these binary frames and immediately forwards the frame to *all* connected clients whose 3‑D distance 
to the sender is ≤ 5 meters.  
 - The server should *not* persist audio; it may keep at most **one** frame per sender as a tiny jitter buffer (configurable 
via a constant).  
 - No authentication, logging, or external services beyond Ratchet.  
 - The server must be able to run via `php server.php` and listen on port **8080**.  
 - The client JavaScript should connect to `wss://<YOUR_DOMAIN>:8080`, send the control messages, record and stream audio, 
and play any incoming binary frames.  
 - The client should expose a function `sendPosition()` that reads the Three.js camera’s position and sends it.  
 - The client should use `MediaRecorder` to record and stream immediately (no buffering).  
 - Use only standard browser APIs (no external JS libraries).  
 - The code must be compatible with PHP 7.3+ and modern browsers (Chrome, Edge, Firefox, Safari 13+).  
 - Include comments in both PHP and JS that explain each logical block.  
 
 ### Deliverables
 1. **server.php** – full Ratchet server code, with constants for `CHAT_RADIUS`, `BUFFER_FRAMES`, and port.  
 2. **client.js** – full client‑side code that:
    - opens the WebSocket, sends register and position, streams audio, and plays incoming frames.  
    - assumes a global variable `userHash` (the unique user ID) is already defined, and a global `camera` object (Three.js 
camera).  
 3. Brief README section (≤ 2 sentences) explaining how to run the server and integrate the client into a Three.js scene.  
 
 ### Constraints
 - No external PHP or JS libraries beyond Ratchet (for PHP) and native browser APIs (for JS).  
 - Code must compile on PHP 7.3+ without extra extensions.  
 - No hard‑coded credentials; the agent must note that the server can be extended with JWT authentication later.  
 - The buffer logic should be optional via `BUFFER_FRAMES` constant; default to 0.  
 - The server must handle binary messages correctly; use `binaryType = 'arraybuffer'` on the client.  
 
 ### Expected Output Format
 Provide each file content separately, prefixed by the file name and a brief comment block.  
 Example:
 ```php
 // server.php
 <?php
 ...
 ```
 ```js
 // client.js
 const socket = new WebSocket('wss://...');
 ...
 ```
 
 **Please generate the code**.  Make sure the PHP uses Ratchet’s `MessageComponentInterface`, and the JS uses only the 
standard WebSocket and MediaRecorder APIs.  
 ```

"""

result = run_agent(request)
print("\nFINAL RESPONSE\n")
print(result)



--- iteration 1 ---
Tool calls: [{'name': 'list_files', 'args': {}, 'id': '5ab3870b-c6dc-4ba5-b1d8-ea540f08d882', 'type': 'tool_call'}]
Executing: list_files({})
(workspace is empty)

--- iteration 2 ---
Tool calls: [{'name': 'write_file', 'args': {'path': 'server.php', 'content': '<?php\n// server.php\n// Lightweight WebSocket proximity chat server using Ratchet.\n// Listens on port 8080 and forwards audio frames to clients within CHAT_RADIUS.\n// No persistence; all state kept in memory.\n\nrequire __DIR__ . \'/vendor/autoload.php\';\n\nuse Ratchet\\ConnectionInterface;\nuse Ratchet\\MessageComponentInterface;\nuse Ratchet\\Server\\IoServer;\nuse Ratchet\\WebSocket\\WsServer;\nuse Ratchet\\Http\\HttpServer;\n\n// Configuration constants\nconst CHAT_RADIUS = 5.0;          // meters\nconst BUFFER_FRAMES = 0;          // number of frames to buffer per sender (0 = no buffering)\nconst PORT = 8080;\n\n/**\n * ProximityChat implements Ratchet\'s MessageComponentInterface.\n * It manages c

In [54]:
print(list_files.invoke({}))


README.md
client.js
server.php


In [55]:
print(read_file.invoke({"path": "README.md"}))


# README

**Proximity Chat** – a lightweight WebSocket‑based proximity chat system for Three.js / WebXR VR worlds.

*Run the server:* `php server.php` (listens on port 8080).  Include `client.js` in your scene and set global `userHash` and `camera` before the script runs.



In [31]:
print(run_command.invoke({
    "command": "python -m compileall app"
}))

exit_code: 0
stdout:
Listing 'app'...

stderr:



In [ ]:
def interactive_agent():
    print("Coding agent ready.")
    print(f"Workspace: {WORKSPACE}")
    print("Type 'exit' or 'quit' to stop.\n")

    while True:
        try:
            request = input("You> ").strip()
        except EOFError:
            break

        if request.lower() in {"exit", "quit"}:
            break

        if not request:
            continue

        answer = run_agent(request, verbose=True)
        print(f"\nAgent> {answer}\n")
